# Data Ingestion, Cleaning & Preprocessing with Pandas

**Dataset:** Retail Sales transactions (raw export from an e-commerce order system)

**Goal:** Take a messy, real-world business dataset (10,000+ rows) containing missing
values, duplicate records, and inconsistent data types, and clean it thoroughly.

**Steps covered:**
1. Load raw dataset into Pandas
2. Identify and impute/handle missing values, outliers, and incorrect data types
3. Feature engineering (extract month/year from dates, calculate profit margins)
4. Export the clean, standardized dataset as `clean_dataset.csv`


## 1. Load raw dataset into Pandas

In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)

raw_path = "raw_retail_sales.csv"
df = pd.read_csv(raw_path)

print(f"Shape: {df.shape[0]} rows x {df.shape[1]} columns")
df.head(10)


Shape: 10800 rows x 13 columns


,OrderID,OrderDate,Category,Product,Region,Channel,PaymentMethod,Quantity,UnitPrice,CostPrice,Revenue,CustomerAge,CustomerRating
0,ORD110433,05-26-2023,toys,Action Figure,North,In-Store,Credit Card,7.0,250.84,154.56,1755.88,34.0,1.0
1,ORD105861,03/02/2024,Beauty,Perfume,Central,Online,UPI,4.0,67.34,32.03,269.36,60.0,3.0
2,ORD107869,2024-03-11,Home & Kitchen,Blender,South,Online,Net Banking,7.0,364.32,152.05,2550.24,NaN,5.0
3,ORD107371,2023-12-27,Electronics,USB-C Hub,North,In-Store,Cash,9.0,489.82,217.55,4408.38,32.0,4.0
4,ORD106699,21/07/2024,HOME & KITCHEN,Bedsheet Set,South,Online,Net Banking,6.0,174.52,NaN,1047.12,999.0,5.0
5,ORD109413,11-27-2024,BEAUTY,Sunscreen,East,Online,Net Banking,7.0,236.02,148.17,1652.14,34.0,2.0
6,ORD107109,03-Jul-2024,beauty,Lipstick,North,In-Store,Credit Card,9.0,73.71,31.81,663.39,37.0,3.0
7,ORD100488,2023-08-22,SPORTS,Dumbbell Set,East,Online,Credit Card,6.0,268.81,142.40,1612.86,52.0,4.0
8,ORD106579,18-Dec-2024,Beauty,Face Serum,Central,Online,UPI,6.0,414.21,166.10,2485.26,33.0,3.0
9,ORD102838,19-Nov-2024,sports,Cricket Bat,North,In-Store,Net Banking,9.0,NaN,205.21,3478.95,30.0,2.0


In [2]:
# Quick structural overview
df.info()


<class 'pandas.DataFrame'>
RangeIndex: 10800 entries, 0 to 10799
Data columns (total 13 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   OrderID         10800 non-null  str    
 1   OrderDate       10693 non-null  str    
 2   Category        10480 non-null  str    
 3   Product         10800 non-null  str    
 4   Region          10579 non-null  str    
 5   Channel         10800 non-null  str    
 6   PaymentMethod   10371 non-null  str    
 7   Quantity        10584 non-null  float64
 8   UnitPrice       10584 non-null  str    
 9   CostPrice       10479 non-null  float64
 10  Revenue         10800 non-null  float64
 11  CustomerAge     10262 non-null  float64
 12  CustomerRating  9938 non-null   float64
dtypes: float64(5), str(8)
memory usage: 1.1 MB


## 2. Initial data quality audit ("before" snapshot)

Before touching anything, capture a baseline of the problems: missing values,
duplicate rows, data types, and inconsistent category labels.

In [3]:
# --- Missing values per column ---
missing_summary = df.isna().sum().to_frame("missing_count")
missing_summary["missing_pct"] = (missing_summary["missing_count"] / len(df) * 100).round(2)
missing_summary.sort_values("missing_count", ascending=False)


,missing_count,missing_pct
CustomerRating,862,7.98
CustomerAge,538,4.98
PaymentMethod,429,3.97
CostPrice,321,2.97
Category,320,2.96
Region,221,2.05
Quantity,216,2.00
UnitPrice,216,2.00
OrderDate,107,0.99
Product,0,0.00


In [4]:
# --- Duplicate rows ---
n_duplicates = df.duplicated().sum()
print(f"Fully duplicated rows: {n_duplicates}")

# Also check duplicates by OrderID (should be a unique key)
dup_order_ids = df.duplicated(subset=["OrderID"]).sum()
print(f"Duplicate OrderIDs: {dup_order_ids}")


Fully duplicated rows: 300
Duplicate OrderIDs: 300


In [5]:
# --- Data types (note Quantity/UnitPrice are 'object' -> should be numeric) ---
df.dtypes


OrderID               str
OrderDate             str
Category              str
Product               str
Region                str
Channel               str
PaymentMethod         str
Quantity          float64
UnitPrice             str
CostPrice         float64
Revenue           float64
CustomerAge       float64
CustomerRating    float64
dtype: object

In [6]:
# --- Inconsistent category labels (casing / whitespace) ---
print("Raw unique values in 'Category':")
print(sorted(df['Category'].dropna().unique()))


Raw unique values in 'Category':
[' Beauty', ' Clothing', ' Electronics', ' Home & Kitchen', ' Sports', ' Toys', 'BEAUTY', 'Beauty', 'CLOTHING', 'Clothing', 'ELECTRONICS', 'Electronics', 'HOME & KITCHEN', 'Home & Kitchen', 'SPORTS', 'Sports', 'TOYS', 'Toys', 'beauty ', 'clothing ', 'electronics ', 'home & kitchen ', 'sports ', 'toys ']


In [7]:
# --- Inconsistent date formats in 'OrderDate' ---
df['OrderDate'].dropna().sample(10, random_state=1).tolist()


['2024-05-22',
 '24/04/2024',
 '23-Sep-2024',
 '05-06-2023',
 '26/02/2024',
 '18/05/2023',
 '10-Sep-2024',
 '11/02/2024',
 '25/01/2023',
 '20/03/2023']

In [8]:
# --- Outlier scan on numeric-ish columns ---
df['CustomerAge'].describe()


count    10262.000000
mean        41.387741
std         56.868977
min         -5.000000
25%         29.000000
50%         38.000000
75%         46.000000
max        999.000000
Name: CustomerAge, dtype: float64

In [9]:
df['Revenue'].describe()


count     10800.000000
mean       1523.047289
std        3785.766434
min           5.120000
25%         448.265000
50%        1078.095000
75%        2073.727500
max      180138.082038
Name: Revenue, dtype: float64

## 3. Cleaning: data types, text standardization, dates

In [10]:
df_clean = df.copy()

# --- 3a. Clean OrderID / drop exact duplicate rows ---
before = len(df_clean)
df_clean = df_clean.drop_duplicates()
print(f"Dropped {before - len(df_clean)} fully duplicated rows")

# Also drop duplicate OrderIDs, keeping the first occurrence
before = len(df_clean)
df_clean = df_clean.drop_duplicates(subset=["OrderID"], keep="first")
print(f"Dropped {before - len(df_clean)} duplicate OrderID rows")


Dropped 300 fully duplicated rows
Dropped 0 duplicate OrderID rows


In [11]:
# --- 3b. Standardize Category text (strip whitespace, title-case) ---
df_clean['Category'] = df_clean['Category'].str.strip().str.title()
df_clean['Category'] = df_clean['Category'].replace({'Home & Kitchen': 'Home & Kitchen'})
sorted(df_clean['Category'].dropna().unique())


['Beauty', 'Clothing', 'Electronics', 'Home & Kitchen', 'Sports', 'Toys']

In [12]:
# --- 3c. Clean Quantity: strip currency/whitespace artifacts, cast to numeric ---
df_clean['Quantity'] = (
    df_clean['Quantity']
    .astype(str)
    .str.replace(r'[^0-9.]', '', regex=True)
    .replace('', np.nan)
    .astype(float)
)

# --- 3d. Clean UnitPrice: strip '$' and other symbols, cast to numeric ---
df_clean['UnitPrice'] = (
    df_clean['UnitPrice']
    .astype(str)
    .str.replace(r'[^0-9.]', '', regex=True)
    .replace('', np.nan)
    .astype(float)
)

df_clean[['Quantity', 'UnitPrice']].dtypes


Quantity     float64
UnitPrice    float64
dtype: object

In [13]:
# --- 3e. Standardize OrderDate: parse multiple formats into a single datetime dtype ---
def parse_mixed_date(value):
    if pd.isna(value):
        return pd.NaT
    for fmt in ("%Y-%m-%d", "%d/%m/%Y", "%m-%d-%Y", "%d-%b-%Y"):
        try:
            return pd.to_datetime(value, format=fmt)
        except (ValueError, TypeError):
            continue
    # Fall back to pandas' flexible parser as a last resort
    return pd.to_datetime(value, errors="coerce", dayfirst=True)

df_clean['OrderDate'] = df_clean['OrderDate'].apply(parse_mixed_date)
df_clean['OrderDate'].sample(10, random_state=1)


4614   2023-11-11
8595   2023-03-23
3921   2024-12-06
7284   2023-12-24
4167   2024-11-08
9698   2023-12-31
7148   2023-02-23
2932   2023-09-26
1626   2023-09-03
3342   2024-07-09
Name: OrderDate, dtype: datetime64[us]

In [14]:
# --- 3f. Standardize text columns (Region, PaymentMethod, Channel) ---
for col in ['Region', 'PaymentMethod', 'Channel', 'Product']:
    df_clean[col] = df_clean[col].astype(str).str.strip().str.title()
    df_clean.loc[df_clean[col] == 'Nan', col] = np.nan

df_clean[['Region', 'PaymentMethod', 'Channel']].apply(lambda s: sorted(s.dropna().unique()))


Region                         [Central, East, North, South, West]
PaymentMethod    [Cash, Credit Card, Debit Card, Net Banking, Upi]
Channel                                         [In-Store, Online]
dtype: object

## 4. Handling outliers

In [15]:
# --- CustomerAge: physically impossible values (e.g. negative, >100) treated as invalid ---
invalid_age_mask = (df_clean['CustomerAge'] < 18) | (df_clean['CustomerAge'] > 100)
print(f"Invalid ages found: {invalid_age_mask.sum()}")
df_clean.loc[invalid_age_mask, 'CustomerAge'] = np.nan


Invalid ages found: 103


In [16]:
# --- Revenue: cap extreme outliers using the IQR method rather than dropping rows ---
Q1 = df_clean['Revenue'].quantile(0.25)
Q3 = df_clean['Revenue'].quantile(0.75)
IQR = Q3 - Q1
upper_bound = Q3 + 1.5 * IQR
lower_bound = max(0, Q1 - 1.5 * IQR)

n_outliers = ((df_clean['Revenue'] < lower_bound) | (df_clean['Revenue'] > upper_bound)).sum()
print(f"Revenue outliers detected (IQR method): {n_outliers}")
print(f"Bounds: [{lower_bound:.2f}, {upper_bound:.2f}]")

df_clean['Revenue'] = df_clean['Revenue'].clip(lower=lower_bound, upper=upper_bound)


Revenue outliers detected (IQR method): 144
Bounds: [0.00, 4512.05]


## 5. Handling missing values

Strategy per column:
- **Category / Region / PaymentMethod / Channel** (categorical): impute with the column mode
- **CustomerAge**: impute with the median (robust to skew/outliers)
- **CustomerRating**: impute with the median rating
- **Quantity / UnitPrice / CostPrice**: impute with the median, since these drive revenue math
- **OrderDate**: rows with unparseable/missing dates are dropped (a transaction without a date isn't usable for time-based analysis)


In [17]:
# Drop rows with missing OrderDate (can't be reliably imputed for transactional data)
before = len(df_clean)
df_clean = df_clean.dropna(subset=['OrderDate'])
print(f"Dropped {before - len(df_clean)} rows with missing/unparseable OrderDate")


Dropped 105 rows with missing/unparseable OrderDate


In [18]:
# Impute categorical columns with the mode
for col in ['Category', 'Region', 'PaymentMethod', 'Channel']:
    mode_val = df_clean[col].mode(dropna=True)[0]
    df_clean[col] = df_clean[col].fillna(mode_val)

# Impute numeric columns with the median
for col in ['CustomerAge', 'CustomerRating', 'Quantity', 'UnitPrice', 'CostPrice']:
    median_val = df_clean[col].median()
    df_clean[col] = df_clean[col].fillna(median_val)

print("Remaining missing values:")
df_clean.isna().sum()


Remaining missing values:


OrderID           0
OrderDate         0
Category          0
Product           0
Region            0
Channel           0
PaymentMethod     0
Quantity          0
UnitPrice         0
CostPrice         0
Revenue           0
CustomerAge       0
CustomerRating    0
dtype: int64

In [19]:
# Fix dtypes now that everything is populated
df_clean['Quantity'] = df_clean['Quantity'].round().astype(int)
df_clean['CustomerRating'] = df_clean['CustomerRating'].round().astype(int)
df_clean['CustomerAge'] = df_clean['CustomerAge'].round().astype(int)
df_clean['UnitPrice'] = df_clean['UnitPrice'].round(2)
df_clean['CostPrice'] = df_clean['CostPrice'].round(2)

# Recompute Revenue consistently from cleaned Quantity/UnitPrice
df_clean['Revenue'] = (df_clean['Quantity'] * df_clean['UnitPrice']).round(2)

df_clean.dtypes


OrderID                      str
OrderDate         datetime64[us]
Category                     str
Product                      str
Region                       str
Channel                      str
PaymentMethod                str
Quantity                   int64
UnitPrice                float64
CostPrice                float64
Revenue                  float64
CustomerAge                int64
CustomerRating             int64
dtype: object

## 6. Feature engineering

In [20]:
# --- Extract Year / Month / Month name from OrderDate ---
df_clean['OrderYear'] = df_clean['OrderDate'].dt.year
df_clean['OrderMonth'] = df_clean['OrderDate'].dt.month
df_clean['OrderMonthName'] = df_clean['OrderDate'].dt.strftime('%B')

# --- Total cost and profit metrics ---
df_clean['TotalCost'] = (df_clean['CostPrice'] * df_clean['Quantity']).round(2)
df_clean['Profit'] = (df_clean['Revenue'] - df_clean['TotalCost']).round(2)
df_clean['ProfitMarginPct'] = (
    (df_clean['Profit'] / df_clean['Revenue']).replace([np.inf, -np.inf], np.nan) * 100
).round(2)

df_clean[['OrderDate', 'OrderYear', 'OrderMonth', 'OrderMonthName',
          'Revenue', 'TotalCost', 'Profit', 'ProfitMarginPct']].head(10)


,OrderDate,OrderYear,OrderMonth,OrderMonthName,Revenue,TotalCost,Profit,ProfitMarginPct
0,2023-05-26,2023,5,May,1755.88,1081.92,673.96,38.38
1,2024-02-03,2024,2,February,269.36,128.12,141.24,52.44
2,2024-03-11,2024,3,March,2550.24,1064.35,1485.89,58.26
3,2023-12-27,2023,12,December,4408.38,1957.95,2450.43,55.59
4,2024-07-21,2024,7,July,1047.12,875.52,171.60,16.39
5,2024-11-27,2024,11,November,1652.14,1037.19,614.95,37.22
6,2024-07-03,2024,7,July,663.39,286.29,377.10,56.84
7,2023-08-22,2023,8,August,1612.86,854.40,758.46,47.03
8,2024-12-18,2024,12,December,2485.26,996.60,1488.66,59.90
9,2024-11-19,2024,11,November,2262.69,1846.89,415.80,18.38


In [21]:
# Sanity check: profit margin distribution looks reasonable (no divide-by-zero blowups)
df_clean['ProfitMarginPct'].describe()


count    10395.000000
mean        38.087202
std         43.259756
min      -2394.360000
25%         29.420000
50%         39.900000
75%         50.310000
max         98.620000
Name: ProfitMarginPct, dtype: float64

## 7. Final validation ("after" snapshot)

In [22]:
print("Shape after cleaning:", df_clean.shape)
print()
print("Missing values remaining:")
print(df_clean.isna().sum().sum())
print()
print("Duplicate rows remaining:", df_clean.duplicated().sum())
print()
print("Dtypes:")
df_clean.dtypes


Shape after cleaning: (10395, 19)

Missing values remaining:
0

Duplicate rows remaining: 0

Dtypes:


OrderID                       str
OrderDate          datetime64[us]
Category                      str
Product                       str
Region                        str
Channel                       str
PaymentMethod                 str
Quantity                    int64
UnitPrice                 float64
CostPrice                 float64
Revenue                   float64
CustomerAge                 int64
CustomerRating              int64
OrderYear                   int32
OrderMonth                  int32
OrderMonthName                str
TotalCost                 float64
Profit                    float64
ProfitMarginPct           float64
dtype: object

In [23]:
df_clean.describe(include='all').T


,count,unique,top,freq,mean,min,25%,50%,75%,max,std
OrderID,10395,10395,ORD110433,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
OrderDate,10395,NaN,NaN,NaN,2024-01-06 02:53:34.545454,2023-01-01 00:00:00,2023-07-07 00:00:00,2024-01-10 00:00:00,2024-07-08 12:00:00,2024-12-31 00:00:00,NaN
Category,10395,6,Electronics,2033,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Product,10395,30,Usb-C Hub,393,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Region,10395,5,East,2289,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Channel,10395,2,Online,5289,NaN,NaN,NaN,NaN,NaN,NaN,NaN
PaymentMethod,10395,5,Credit Card,2489,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Quantity,10395.0,NaN,NaN,NaN,5.569697,1.0,3.0,6.0,8.0,10.0,2.835498
UnitPrice,10395.0,NaN,NaN,NaN,251.244187,5.0,131.055,251.41,368.925,499.96,140.396242
CostPrice,10395.0,NaN,NaN,NaN,150.743837,2.15,75.6,145.92,213.515,397.1,89.684153


In [24]:
df_clean.head(10)


,OrderID,OrderDate,Category,Product,Region,Channel,PaymentMethod,Quantity,UnitPrice,CostPrice,Revenue,CustomerAge,CustomerRating,OrderYear,OrderMonth,OrderMonthName,TotalCost,Profit,ProfitMarginPct
0,ORD110433,2023-05-26,Toys,Action Figure,North,In-Store,Credit Card,7,250.84,154.56,1755.88,34,1,2023,5,May,1081.92,673.96,38.38
1,ORD105861,2024-02-03,Beauty,Perfume,Central,Online,Upi,4,67.34,32.03,269.36,60,3,2024,2,February,128.12,141.24,52.44
2,ORD107869,2024-03-11,Home & Kitchen,Blender,South,Online,Net Banking,7,364.32,152.05,2550.24,38,5,2024,3,March,1064.35,1485.89,58.26
3,ORD107371,2023-12-27,Electronics,Usb-C Hub,North,In-Store,Cash,9,489.82,217.55,4408.38,32,4,2023,12,December,1957.95,2450.43,55.59
4,ORD106699,2024-07-21,Home & Kitchen,Bedsheet Set,South,Online,Net Banking,6,174.52,145.92,1047.12,38,5,2024,7,July,875.52,171.60,16.39
5,ORD109413,2024-11-27,Beauty,Sunscreen,East,Online,Net Banking,7,236.02,148.17,1652.14,34,2,2024,11,November,1037.19,614.95,37.22
6,ORD107109,2024-07-03,Beauty,Lipstick,North,In-Store,Credit Card,9,73.71,31.81,663.39,37,3,2024,7,July,286.29,377.10,56.84
7,ORD100488,2023-08-22,Sports,Dumbbell Set,East,Online,Credit Card,6,268.81,142.40,1612.86,52,4,2023,8,August,854.40,758.46,47.03
8,ORD106579,2024-12-18,Beauty,Face Serum,Central,Online,Upi,6,414.21,166.10,2485.26,33,3,2024,12,December,996.60,1488.66,59.90
9,ORD102838,2024-11-19,Sports,Cricket Bat,North,In-Store,Net Banking,9,251.41,205.21,2262.69,30,2,2024,11,November,1846.89,415.80,18.38


## 8. Before / after comparison summary

In [25]:
summary = pd.DataFrame({
    "Metric": [
        "Rows",
        "Duplicate rows",
        "Missing values (total)",
        "Quantity dtype",
        "UnitPrice dtype",
        "OrderDate dtype",
        "Unique Category labels",
    ],
    "Before": [
        len(df),
        df.duplicated().sum(),
        df.isna().sum().sum(),
        df['Quantity'].dtype,
        df['UnitPrice'].dtype,
        df['OrderDate'].dtype,
        df['Category'].dropna().nunique(),
    ],
    "After": [
        len(df_clean),
        df_clean.duplicated().sum(),
        df_clean.isna().sum().sum(),
        df_clean['Quantity'].dtype,
        df_clean['UnitPrice'].dtype,
        df_clean['OrderDate'].dtype,
        df_clean['Category'].dropna().nunique(),
    ],
})
summary


,Metric,Before,After
0,Rows,10800,10395
1,Duplicate rows,300,0
2,Missing values (total),3230,0
3,Quantity dtype,float64,int64
4,UnitPrice dtype,str,float64
5,OrderDate dtype,str,datetime64[us]
6,Unique Category labels,24,6


## 9. Export the clean, standardized dataset

In [26]:
output_path = "clean_dataset.csv"
df_clean.to_csv(output_path, index=False)
print(f"Saved cleaned dataset to '{output_path}' with {df_clean.shape[0]} rows and {df_clean.shape[1]} columns.")


Saved cleaned dataset to 'clean_dataset.csv' with 10395 rows and 19 columns.
